# EchoPilot Research 01: Document Parsing & Semantic Chunking

This notebook implements the offline data preparation pipeline for EchoPilot:
1. Extracting clean textual content from uploaded educational PDFs (`pypdf` / `pymupdf`).
2. Preserving page numbers, headers, and section boundaries.
3. Generating sliding semantic chunks (300-500 words, 50-word overlap) to fit small local LLM context windows.
4. Constructing contrastive retrieval pairs and instruction-tuning records.


In [ ]:
import os
import sys
import json
import glob
import re
import pandas as pd

# Add src to python path
sys.path.append(os.path.abspath("../src"))
from chunker import extract_text_from_pdf, chunk_document

print("Chunker module successfully loaded.")


## Step 1: Scan Existing PDF Uploads
Let's inspect the files currently present in EchoPilot's upload storage (`../../server/data/uploads`).


In [ ]:
upload_dir = os.path.abspath("../../server/data/uploads")
pdf_files = glob.glob(os.path.join(upload_dir, "*.pdf"))
print(f"Found {len(pdf_files)} PDF files in storage:")
for p in pdf_files:
    print(f" - {os.path.basename(p)} ({os.path.getsize(p) / 1024:.1f} KB)")


## Step 2: Page-Aware Text Extraction
Extract text per page while preserving page numbers and formatting cues.


In [ ]:
all_chunks = []

for pdf_path in pdf_files:
    fname = os.path.basename(pdf_path)
    print(f"Processing: {fname}...")
    try:
        pages_data = extract_text_from_pdf(pdf_path)
        print(f"  Extracted {len(pages_data)} pages.")
        
        chunks = chunk_document(
            pages_data=pages_data,
            chunk_size=350,
            chunk_overlap=50,
            material_id=fname.replace(".pdf", ""),
            material_name=fname
        )
        print(f"  Created {len(chunks)} semantic chunks.")
        all_chunks.extend(chunks)
    except Exception as e:
        print(f"  Error reading {fname}: {e}")

print(f"Total chunks generated: {len(all_chunks)}")


## Step 3: Inspect Chunk Structure & Statistics
Check chunk word counts, page attribution, and metadata.


In [ ]:
if all_chunks:
    df_chunks = pd.DataFrame(all_chunks)
    print(df_chunks[["chunk_id", "material_name", "page_number", "section_heading", "word_count"]].head(8))
    print("
Summary Statistics of Chunk Word Counts:")
    print(df_chunks["word_count"].describe())
else:
    print("No chunks extracted yet. Run with sample PDFs.")


## Step 4: Export Chunks for Indexing
Save the generated chunks into JSON Lines format for local retrieval in Notebook 02.


In [ ]:
out_file = "../data/extracted_chunks.jsonl"
os.makedirs(os.path.dirname(out_file), exist_ok=True)

with open(out_file, "w", encoding="utf-8") as f:
    for c in all_chunks:
        f.write(json.dumps(c) + "
")

print(f"Exported {len(all_chunks)} chunks to {out_file}")
